In [1]:
import pandas as pd
import numpy as np
import joblib

# Load the regression dataset
data_path = "../data/processed/CarbonOpt_XGBoost_Regression.xlsx"

df = pd.read_excel(data_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (3162, 11)

Columns:
['state', 'year', 'residential_billion_btu', 'commercial_billion_btu', 'industrial_billion_btu', 'transportation_billion_btu', 'co2_lag_1', 'co2_lag_2', 'energy_growth_rate', 'co2_growth_rate', 'co2_next_year']


In [2]:
regression_model = joblib.load("../models/xgb_regression.pkl")
classification_model = joblib.load("../models/xgb_classification.pkl")
risk_encoder = joblib.load("../models/risk_label_encoder.pkl")
isolation_forest = joblib.load("../models/isolation_forest.pkl")
isolation_scaler = joblib.load("../models/isolation_scaler.pkl")
kmeans_model = joblib.load("../models/kmeans_clustering.pkl")
kmeans_scaler = joblib.load("../models/kmeans_scaler.pkl")

print("All trained models loaded successfully.")

All trained models loaded successfully.


In [3]:
print("Regression model:", type(regression_model))
print("Classification model:", type(classification_model))
print("Risk encoder:", type(risk_encoder))
print("Isolation Forest:", type(isolation_forest))
print("K-Means:", type(kmeans_model))

Regression model: <class 'sklearn.pipeline.Pipeline'>
Classification model: <class 'sklearn.pipeline.Pipeline'>
Risk encoder: <class 'sklearn.preprocessing._label.LabelEncoder'>
Isolation Forest: <class 'sklearn.ensemble._iforest.IsolationForest'>
K-Means: <class 'sklearn.cluster._kmeans.KMeans'>


In [4]:
feature_columns = [
    "state",
    "year",
    "residential_billion_btu",
    "commercial_billion_btu",
    "industrial_billion_btu",
    "transportation_billion_btu",
    "co2_lag_1",
    "co2_lag_2",
    "energy_growth_rate",
    "co2_growth_rate"
]

In [5]:
# Prepare regression input
X_reg = df[feature_columns].copy()

# Predict next-year CO2
df["predicted_co2_next_year"] = regression_model.predict(X_reg)

print("Regression predictions generated.")
print(df[
    [
        "state",
        "year",
        "co2_next_year",
        "predicted_co2_next_year"
    ]
].head(10))

Regression predictions generated.
  state  year  co2_next_year  predicted_co2_next_year
0    AK  1962          5.366                 5.628583
1    AK  1963          5.620                 5.733435
2    AK  1964          5.720                 5.996279
3    AK  1965          6.812                 6.296746
4    AK  1966          7.801                 7.284564
5    AK  1967          8.170                 9.039012
6    AK  1968         10.014                 8.806603
7    AK  1969         11.356                12.375397
8    AK  1970         12.643                12.178847
9    AK  1971         13.432                13.314465


In [6]:
X_class = df[feature_columns].copy()

In [7]:
# Predict numerical class
risk_pred_encoded = classification_model.predict(X_class)

# Convert numerical class back to labels
df["risk_class"] = risk_encoder.inverse_transform(
    risk_pred_encoded.astype(int)
)

print("Risk classification generated.")

print(
    df[
        [
            "state",
            "year",
            "predicted_co2_next_year",
            "risk_class"
        ]
    ].head(10)
)

Risk classification generated.
  state  year  predicted_co2_next_year risk_class
0    AK  1962                 5.628583        Low
1    AK  1963                 5.733435        Low
2    AK  1964                 5.996279        Low
3    AK  1965                 6.296746        Low
4    AK  1966                 7.284564        Low
5    AK  1967                 9.039012        Low
6    AK  1968                 8.806603        Low
7    AK  1969                12.375397        Low
8    AK  1970                12.178847        Low
9    AK  1971                13.314465        Low


In [8]:
print("\nRisk distribution:")
print(df["risk_class"].value_counts())


Risk distribution:
risk_class
Medium    1128
High      1025
Low       1009
Name: count, dtype: int64


In [9]:
# Features used by Isolation Forest
if_features = [
    "residential_billion_btu",
    "commercial_billion_btu",
    "industrial_billion_btu",
    "transportation_billion_btu",
    "co2_million_metric_tons",
    "co2_per_energy",
    "energy_growth_rate",
    "co2_growth_rate"
]

X_if = df[if_features].copy()

# Apply the SAME scaler used during training
X_if_scaled = isolation_scaler.transform(X_if)

# Predict anomalies
if_prediction = isolation_forest.predict(X_if_scaled)

# Convert sklearn output:
#  1  = normal
# -1  = anomaly
df["anomaly_flag"] = np.where(
    if_prediction == -1,
    "Anomaly",
    "Normal"
)

# Anomaly score
df["anomaly_score"] = isolation_forest.decision_function(X_if_scaled)

print("Isolation Forest integrated.")
print("\nAnomaly distribution:")
print(df["anomaly_flag"].value_counts())

KeyError: "['co2_million_metric_tons', 'co2_per_energy'] not in index"

In [10]:
# Load the dataset used for Isolation Forest
if_data = pd.read_excel(
    "../data/processed/CarbonOpt_IsolationForest.xlsx"
)

print("Isolation Forest dataset shape:", if_data.shape)
print(if_data.columns.tolist())

Isolation Forest dataset shape: (3162, 10)
['state', 'year', 'residential_billion_btu', 'commercial_billion_btu', 'industrial_billion_btu', 'transportation_billion_btu', 'co2_million_metric_tons', 'co2_per_energy', 'energy_growth_rate', 'co2_growth_rate']


In [11]:
if_features = [
    "residential_billion_btu",
    "commercial_billion_btu",
    "industrial_billion_btu",
    "transportation_billion_btu",
    "co2_million_metric_tons",
    "co2_per_energy",
    "energy_growth_rate",
    "co2_growth_rate"
]

X_if = if_data[if_features].copy()

X_if_scaled = isolation_scaler.transform(X_if)

if_prediction = isolation_forest.predict(X_if_scaled)

if_data["anomaly_flag"] = np.where(
    if_prediction == -1,
    "Anomaly",
    "Normal"
)

if_data["anomaly_score"] = (
    isolation_forest.decision_function(X_if_scaled)
)

print("Isolation Forest integrated.")
print(if_data["anomaly_flag"].value_counts())

Isolation Forest integrated.
anomaly_flag
Normal     3003
Anomaly     159
Name: count, dtype: int64


In [12]:
# Prepare K-Means input
X_kmeans = if_data[if_features].copy()

# Apply the SAME scaler used during K-Means training
X_kmeans_scaled = kmeans_scaler.transform(X_kmeans)

# Predict cluster
if_data["cluster"] = kmeans_model.predict(X_kmeans_scaled)

print("K-Means integrated.")

print("\nCluster distribution:")
print(if_data["cluster"].value_counts())

print("\nSample:")
print(
    if_data[
        ["state", "year", "cluster", "anomaly_flag", "anomaly_score"]
    ].head(10)
)

K-Means integrated.

Cluster distribution:
cluster
0    2553
1     609
Name: count, dtype: int64

Sample:
  state  year  cluster anomaly_flag  anomaly_score
0    AK  1962        0       Normal       0.078955
1    AK  1963        0       Normal       0.118611
2    AK  1964        0       Normal       0.117015
3    AK  1965        0       Normal       0.124492
4    AK  1966        0      Anomaly      -0.003993
5    AK  1967        0       Normal       0.043867
6    AK  1968        0       Normal       0.129050
7    AK  1969        0       Normal       0.005002
8    AK  1970        0       Normal       0.050750
9    AK  1971        0       Normal       0.104116


In [13]:
# Keep only the outputs generated by Isolation Forest and K-Means
if_outputs = if_data[
    [
        "state",
        "year",
        "anomaly_flag",
        "anomaly_score",
        "cluster"
    ]
].copy()

# Merge with regression + classification dataframe
df_master = df.merge(
    if_outputs,
    on=["state", "year"],
    how="inner"
)

print("Master dataframe shape:", df_master.shape)

print(
    df_master[
        [
            "state",
            "year",
            "predicted_co2_next_year",
            "risk_class",
            "anomaly_flag",
            "anomaly_score",
            "cluster"
        ]
    ].head(10)
)

Master dataframe shape: (3162, 16)
  state  year  predicted_co2_next_year risk_class anomaly_flag  anomaly_score  \
0    AK  1962                 5.628583        Low       Normal       0.078955   
1    AK  1963                 5.733435        Low       Normal       0.118611   
2    AK  1964                 5.996279        Low       Normal       0.117015   
3    AK  1965                 6.296746        Low       Normal       0.124492   
4    AK  1966                 7.284564        Low      Anomaly      -0.003993   
5    AK  1967                 9.039012        Low       Normal       0.043867   
6    AK  1968                 8.806603        Low       Normal       0.129050   
7    AK  1969                12.375397        Low       Normal       0.005002   
8    AK  1970                12.178847        Low       Normal       0.050750   
9    AK  1971                13.314465        Low       Normal       0.104116   

   cluster  
0        0  
1        0  
2        0  
3        0  
4       

In [14]:
import shap

# Use the classification features
X_shap = df_master[feature_columns].copy()

# Extract the preprocessing transformer and XGBoost classifier
preprocessor = classification_model.named_steps["preprocessor"]
classifier = classification_model.named_steps["classifier"]

# Transform the input exactly as during training
X_shap_transformed = preprocessor.transform(X_shap)

print("Transformed SHAP input shape:", X_shap_transformed.shape)

c:\Users\angel\OneDrive\Desktop\carbonOPT\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformed SHAP input shape: (3162, 60)


In [15]:
# Get transformed feature names
feature_names = preprocessor.get_feature_names_out()

# Create SHAP explainer for the trained XGBoost classifier
explainer = shap.TreeExplainer(classifier)

shap_values = explainer.shap_values(X_shap_transformed)

print("SHAP calculation completed.")
print("SHAP output shape:", np.array(shap_values).shape)

SHAP calculation completed.
SHAP output shape: (3162, 60, 3)


In [16]:
# Convert SHAP output to a consistent ndarray
shap_array = np.array(shap_values)

print("SHAP array dimensions:", shap_array.shape)
print("Classes:", risk_encoder.classes_)

SHAP array dimensions: (3162, 60, 3)
Classes: ['High' 'Low' 'Medium']


In [17]:
# ============================================================
# CARBONOPT — FINAL INTEGRATION
# SHAP + OPTIMIZATION + MASTER OUTPUT
# ============================================================

import os
import numpy as np
import pandas as pd
import shap

# ------------------------------------------------------------
# 1. SHAP EXPLANATION
# ------------------------------------------------------------

X_shap = df_master[feature_columns].copy()

preprocessor = classification_model.named_steps["preprocessor"]
classifier = classification_model.named_steps["classifier"]

X_shap_transformed = preprocessor.transform(X_shap)
feature_names = preprocessor.get_feature_names_out()

explainer = shap.TreeExplainer(classifier)
shap_values = explainer.shap_values(X_shap_transformed)

shap_array = np.array(shap_values)

print("SHAP shape:", shap_array.shape)

# ------------------------------------------------------------
# 2. HANDLE MULTICLASS SHAP OUTPUT
# ------------------------------------------------------------

# Find predicted class for every observation
predicted_encoded = classification_model.predict(X_shap)

# SHAP may be:
# (samples, features, classes)
# or (classes, samples, features)

if shap_array.ndim == 3:

    if shap_array.shape[0] == len(df_master):
        # samples, features, classes
        sample_axis = 0
        class_axis = 2

        top_features = []

        for i in range(len(df_master)):
            class_id = int(predicted_encoded[i])

            values = shap_array[i, :, class_id]

            top_idx = np.argmax(np.abs(values))

            top_features.append(
                feature_names[top_idx]
            )

    elif shap_array.shape[1] == len(df_master):
        # classes, samples, features
        top_features = []

        for i in range(len(df_master)):
            class_id = int(predicted_encoded[i])

            values = shap_array[class_id, i, :]

            top_idx = np.argmax(np.abs(values))

            top_features.append(
                feature_names[top_idx]
            )

    else:
        raise ValueError(
            f"Unexpected SHAP shape: {shap_array.shape}"
        )

else:
    raise ValueError(
        f"Unexpected SHAP dimensions: {shap_array.shape}"
    )

df_master["top_risk_driver"] = top_features

# Remove preprocessing prefixes for cleaner presentation
df_master["top_risk_driver"] = (
    df_master["top_risk_driver"]
    .str.replace("numeric__", "", regex=False)
    .str.replace("categorical__", "", regex=False)
)

print("SHAP explanations added.")

# ------------------------------------------------------------
# 3. OPTIMIZATION RESULTS
# ------------------------------------------------------------

# Results from Notebook 06
baseline_co2 = 682.2365
optimized_co2 = 642.51605
target_co2 = 642.7188

optimal_reductions = {
    "residential_reduction_%": 17.001169,
    "commercial_reduction_%": 1.393424,
    "industrial_reduction_%": 12.794114,
    "transportation_reduction_%": 11.247456
}

co2_reduction = baseline_co2 - optimized_co2
reduction_percent = (
    co2_reduction / baseline_co2
) * 100

# ------------------------------------------------------------
# 4. ADD OPTIMIZATION SUMMARY COLUMNS
# ------------------------------------------------------------

df_master["optimized_predicted_co2"] = optimized_co2
df_master["estimated_co2_reduction"] = co2_reduction

for column, value in optimal_reductions.items():
    df_master[column] = value

# ------------------------------------------------------------
# 5. FINAL CARBONOPT OUTPUT
# ------------------------------------------------------------

final_columns = [
    "state",
    "year",

    # Forecast
    "predicted_co2_next_year",

    # Risk
    "risk_class",

    # Anomaly
    "anomaly_flag",
    "anomaly_score",

    # Clustering
    "cluster",

    # Explainability
    "top_risk_driver",

    # Optimization
    "residential_reduction_%",
    "commercial_reduction_%",
    "industrial_reduction_%",
    "transportation_reduction_%",

    "optimized_predicted_co2",
    "estimated_co2_reduction"
]

final_output = df_master[final_columns].copy()

# ------------------------------------------------------------
# 6. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CARBONOPT FINAL INTEGRATION")
print("=" * 70)

print("\nRows:", len(final_output))
print("Columns:", len(final_output.columns))

print("\nRisk distribution:")
print(final_output["risk_class"].value_counts())

print("\nAnomaly distribution:")
print(final_output["anomaly_flag"].value_counts())

print("\nCluster distribution:")
print(final_output["cluster"].value_counts())

print("\nOptimization:")
print(f"Baseline CO2       : {baseline_co2:.4f}")
print(f"Optimized CO2      : {optimized_co2:.4f}")
print(f"CO2 reduction      : {co2_reduction:.4f}")
print(f"Reduction percent  : {reduction_percent:.2f}%")
print(f"Target CO2         : {target_co2:.4f}")

print("\nFinal sample:")
print(final_output.head(10))

# ------------------------------------------------------------
# 7. SAVE FINAL RESULT
# ------------------------------------------------------------

os.makedirs("../outputs/final", exist_ok=True)

output_path = "../outputs/final/CarbonOpt_Master_Output.xlsx"

final_output.to_excel(
    output_path,
    index=False
)

print("\nSaved successfully:")
print(output_path)

SHAP shape: (3162, 60, 3)
SHAP explanations added.

CARBONOPT FINAL INTEGRATION

Rows: 3162
Columns: 14

Risk distribution:
risk_class
Medium    1128
High      1025
Low       1009
Name: count, dtype: int64

Anomaly distribution:
anomaly_flag
Normal     3003
Anomaly     159
Name: count, dtype: int64

Cluster distribution:
cluster
0    2553
1     609
Name: count, dtype: int64

Optimization:
Baseline CO2       : 682.2365
Optimized CO2      : 642.5160
CO2 reduction      : 39.7205
Reduction percent  : 5.82%
Target CO2         : 642.7188

Final sample:
  state  year  predicted_co2_next_year risk_class anomaly_flag  anomaly_score  \
0    AK  1962                 5.628583        Low       Normal       0.078955   
1    AK  1963                 5.733435        Low       Normal       0.118611   
2    AK  1964                 5.996279        Low       Normal       0.117015   
3    AK  1965                 6.296746        Low       Normal       0.124492   
4    AK  1966                 7.284564    